In [1]:
import pandas as pd
import numpy as np 
import re 

In [2]:
city_df = pd.read_csv(r'D:\azure_housing_analytics\dataset\bronze\city_df.csv')
province_df = pd.read_csv(r'D:\azure_housing_analytics\dataset\bronze\province_df.csv')

city_df['provinceCode'] = city_df['provinceCode'].astype(object)


In [3]:
# --- Clean city table ---
dim_city_clean = city_df.copy()
dim_city_clean['cityName'] = dim_city_clean['name'].str.replace(
    r'(?i)^(City of|Science City of|Island Garden City of)\s+', '', regex=True
)
dim_city_clean = dim_city_clean.rename(columns={'code': 'cityCode', 'name': 'cityNameRaw'})
dim_city_clean = dim_city_clean.drop(columns=['oldName', 'psgc10DigitCode', 'districtCode'], errors='ignore')

dim_city_clean['cityName'] = dim_city_clean['cityName'].str.replace(r'(?i)\s+city$', '', regex=True)
dim_city_clean['cityName'] = dim_city_clean['cityName'].str.lower()
dim_city_clean = dim_city_clean.drop(columns=['cityNameRaw'])

metro_manila_province = [
    "manila", "mandaluyong", "marikina", "pasig", "quezon",
    "san juan", "caloocan", "malabon", "navotas", "valenzuela",
    "las piñas", "makati", "muntinlupa", "parañaque", "pasay",
    "pateros", "taguig"
]
dim_city_clean.loc[dim_city_clean['cityName'].isin(metro_manila_province), 'provinceCode'] = "130000000"
dim_city_clean.loc[dim_city_clean['cityName'] == 'isabela', 'provinceCode'] = "150700000"



In [4]:
# --- Clean province table ---
dim_province_clean = province_df.rename(columns={'code': 'provinceCode', 'name': 'provinceName'})
dim_province_clean = dim_province_clean.drop(columns=['psgc10DigitCode'], errors='ignore')

ncr_row = pd.DataFrame([{
    "provinceCode": "130000000",   # changed from 133900000
    "provinceName": "Metro Manila",
    "regionCode": "130000000",
    "islandGroupCode": "luzon"
}])

dim_province_clean = pd.concat([dim_province_clean, ncr_row], ignore_index=True)
dim_province_clean['provinceName'] = dim_province_clean['provinceName'].str.lower()



In [34]:

silver_location = 'D:/Data_Engineering/Housing_Loan_Clone/azure_housing_analytics/dataset/silver/'

dim_city_clean.to_csv(silver_location + 'dim_city.csv', index=False)
dim_province_clean.to_csv(silver_location + 'dim_province.csv', index=False)

In [5]:
merged_geo = dim_city_clean.merge(
    dim_province_clean,
    on='provinceCode',
    how='left',
    suffixes=('_c', '_p')
)

new_table = merged_geo.drop(columns=['islandGroupCode_c','regionCode_p','regionCode_c']) \
            .rename(columns={'regionCode_c':'regionCode','islandGroupCode_p':'islandGroup'})
new_table.head()

,cityCode,isCapital,provinceCode,cityName,provinceName,islandGroup
0,12805000,False,12800000.0,batac,ilocos norte,luzon
1,12812000,True,12800000.0,laoag,ilocos norte,luzon
2,12906000,False,12900000.0,candon,ilocos sur,luzon
3,12934000,True,12900000.0,vigan,ilocos sur,luzon
4,13314000,True,13300000.0,san fernando,la union,luzon


In [40]:
silver_location = 'D:/Data_Engineering/Housing_Loan_Clone/azure_housing_analytics/dataset/silver/'
new_table.to_csv(silver_location + 'dim_geo.csv', index=False)
print('dim_geo.csv table created successfully')

dim_geo.csv table created successfully


In [6]:
## create unified key
dim_geo = pd.read_csv(r'D:\azure_housing_analytics\dataset\silver\dim_geo.csv')

In [7]:
# Filter rows where either cityCode or provinceCode is null/empty
missing_codes = new_table[
    new_table['cityCode'].isnull() | 
    new_table['provinceCode'].isnull()
]

# Display all columns for the filtered rows
display(missing_codes)

,cityCode,isCapital,provinceCode,cityName,provinceName,islandGroup
145,129804000,False,NaN,cotabato,NaN,NaN


In [ ]:
import pandas as pd

# 1. Prepare city records: geographyId takes the raw cityCode
cities = dim_geo.copy()
cities['geographyId'] = cities['cityCode'].astype(str)

# 2. Prepare province-only records: geographyId takes the raw provinceCode
provinces = dim_geo[['provinceCode', 'provinceName', 'islandGroup']].drop_duplicates().copy()
provinces['geographyId'] = provinces['provinceCode'].astype(str)
provinces['cityCode'] = None
provinces['cityName'] = None
provinces['isCapital'] = False

# 3. Combine both into your location dimension
dim_location = pd.concat([cities, provinces], ignore_index=True)

# Reorder columns for clarity
dim_location = dim_location[
    ['geographyId', 'provinceCode', 'cityCode', 'provinceName', 'cityName', 'isCapital', 'islandGroup']
]

dim_location.to_csv(r'D:\azure_housing_analytics\dataset\gold\dim_location.csv', index=False)
print('dim_location saved successfully to gold layer.')
display(dim_location)

dim_location


,geographyId,provinceCode,cityCode,provinceName,cityName,isCapital,islandGroup
0,12805000,12800000,12805000,ilocos norte,batac,False,luzon
1,12812000,12800000,12812000,ilocos norte,laoag,True,luzon
2,12906000,12900000,12906000,ilocos sur,candon,False,luzon
3,12934000,12900000,12934000,ilocos sur,vigan,True,luzon
4,13314000,13300000,13314000,la union,san fernando,True,luzon
...,...,...,...,...,...,...,...
195,160200000,160200000,None,agusan del norte,None,False,mindanao
196,160300000,160300000,None,agusan del sur,None,False,mindanao
197,166700000,166700000,None,surigao del norte,None,False,mindanao
198,166800000,166800000,None,surigao del sur,None,False,mindanao
